# C2 classifier — fine-tune for permission classification on Colab

Launcher for the `mostargate.classifier.train` module. The training code lives in `mostargate/classifier/train.py` in the repo; this notebook just bootstraps a Colab T4 instance and runs that module.

**Workflow:**
1. `Runtime → Change runtime type → T4 GPU` (free tier).
2. `Runtime → Run all`.
3. When training finishes, download `classifier_model_roberta.zip` from the **Files** panel on the left, unzip locally into `dataset/classifier_artifacts/`.

Default config: `roberta-base`, 40 epochs, LR 2e-5, batch 16, fp32. Wall time on T4: ~5–7 minutes total. See `docs/phase_c_classifier_findings.md` §7 for results.

An optional `roberta-large` cell is included below (commented out) — try that if you want to test the higher-capacity variant. Expect ~3× training time and a modest precision/F1 improvement.

## 1. Pull the repo

Always start by `%cd /content` — this resets the working directory so re-runs after a previous clone don't create nested `mostargate/mostargate/` matryoshka folders or fail to remove the directory you're inside.

In [ ]:
%cd /content
!rm -rf mostargate
!git clone https://github.com/0xballistics/mostargate.git
%cd mostargate

## 2. Install Python deps

Colab images already include `torch` and `numpy`. We add the few packages `mostargate.classifier.train` imports beyond that.

In [ ]:
!pip install -q transformers datasets sentencepiece protobuf python-dotenv

## 3. Verify GPU is attached

In [ ]:
!nvidia-smi

## 4. Train roberta-base

Canonical Phase C settings per `docs/phase_c_classifier_findings.md` §7.3:
- 40 epochs, batch 16, LR 2e-5, max_len 256, seed 42
- BCEWithLogitsLoss with per-class `pos_weight` clipped at 10.0
- `max_grad_norm` 1.0
- fp32 (fp16 disabled by default due to a transformers 5.x grad-unscale regression)
- No internal validation hold-out — all 500 train records used for gradient updates.

First run downloads roberta-base (~500 MB) from the HuggingFace hub. Subsequent re-runs in the same session use the cached download.

In [ ]:
!python -m mostargate.classifier.train --epochs 40 --learning-rate 2e-5

## 4b. (Optional) Train roberta-large variant

`roberta-large` is ~3× larger (355M params) than `roberta-base`. It uses the **same architecture and hyperparameters**, so it tolerates LR 2e-5 and the same `pos_weight` settings.

**Expected impact** (versus roberta-base at 40 epochs):
- Training time: ~15 minutes on T4 (vs 5 min for base)
- Macro-F1: likely +2–4 pp (closing some of the gap to Claude's 0.886)
- Undershoot: likely -3–5 pp (the main bottleneck per §7.8 of the findings doc)
- Macro-P: probably similar or slightly higher

**Tradeoff**: 3× larger model means ~3× slower CPU inference at deployment time and ~3× more disk for the saved weights. If results materially exceed roberta-base, document and switch; if marginal, roberta-base remains the canonical model.

Skip cell 4 above (roberta-base training) and uncomment + run this cell instead. The trained model overwrites `dataset/classifier_artifacts/model/`.

In [ ]:
# Uncomment to train roberta-large instead of roberta-base:
# !python -m mostargate.classifier.train --model roberta-large --epochs 40 --learning-rate 2e-5

## 5. Test the trained model - per-permission F1 at threshold 0.5

Loads the saved model, runs inference on the 100-record test set, prints probability distribution and per-permission F1. The `probs` variable produced here is reused by the threshold-sweep cell below.

In [ ]:
import json, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from mostargate.constants import TOOLS

PERMISSIONS = list(TOOLS.keys())
MODEL_DIR  = "dataset/classifier_artifacts/model"

tok   = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).cuda().eval()

test  = json.load(open("dataset/test.json"))
enc   = tok([r["prompt"] for r in test], padding=True, truncation=True,
            max_length=256, return_tensors="pt").to("cuda")
with torch.no_grad():
    probs = torch.sigmoid(model(**enc).logits).cpu().numpy()

print(f"shape={probs.shape}  mean={probs.mean():.3f}  std={probs.std():.3f}\n")
for lo, hi in [(0,.05),(.05,.20),(.20,.40),(.40,.60),(.60,.80),(.80,.95),(.95,1.01)]:
    n   = ((probs >= lo) & (probs < hi)).sum()
    pct = 100*n/probs.size
    print(f"[{lo:.2f}, {hi:.2f}): {pct:5.1f}%  {'#'*int(pct/2)}")

gt = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)
preds = (probs >= 0.5).astype(int)
print(f"\n{'permission':<22}{'gt+':>5}{'pred+':>7}{'TP':>4}{'FP':>4}{'FN':>4}{'P':>6}{'R':>6}{'F1':>6}")
for j, p in enumerate(PERMISSIONS):
    tp = ((gt[:,j]==1) & (preds[:,j]==1)).sum()
    fp = ((gt[:,j]==0) & (preds[:,j]==1)).sum()
    fn = ((gt[:,j]==1) & (preds[:,j]==0)).sum()
    P  = tp / max(tp+fp, 1); R = tp / max(tp+fn, 1)
    F1 = 2*P*R / max(P+R, 1e-9)
    print(f"{p:<22}{gt[:,j].sum():>5}{preds[:,j].sum():>7}{tp:>4}{fp:>4}{fn:>4}{P:>6.2f}{R:>6.2f}{F1:>6.2f}")

macro_f1 = np.mean([
    (lambda tp,fp,fn: 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0)(
        ((gt[:,j]==1) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==0) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==1) & (preds[:,j]==0)).sum(),
    )
    for j in range(15)
])
print(f"\nmacro-F1 at threshold 0.5: {macro_f1:.3f}")
print(f"(reference: TF-IDF=0.70, Claude Haiku=0.89)")

## 6. Threshold sweep — six C2 configurations + JSON archive

Caches the probability matrix from cell 5 to `dataset/classifier_artifacts/roberta_test_probs.npy` (bundled into the zip in cell 8), then defers to `mostargate.classifier.sweep` for the threshold sweep + metrics aggregation.

The CLI module is the single source of truth for thresholds and metric calculation — the Claude Haiku baseline (`mostargate.classifier.baselines`) imports the same `THRESHOLD_CONFIGS` and applier, so the classifier and baselines produce JSON in identical condition/configurations/summary shape. The Phase E comparison report ingests both files uniformly.

Output: `results/<short_model>_sweep.json` (tracked in git). Multiple runs (base, large) write to different filenames and don't overwrite each other.

In [ ]:
import numpy as np
from pathlib import Path

# Cache the probability matrix from cell 5 so the sweep CLI (and the
# zip in cell 8) can pick it up without re-running inference.
Path("dataset/classifier_artifacts").mkdir(parents=True, exist_ok=True)
np.save("dataset/classifier_artifacts/roberta_test_probs.npy", probs)
print(f"Cached probability matrix → roberta_test_probs.npy {probs.shape}\n")

# Apply the six C2 threshold configurations via the shared module.
# Writes results/<short_model>_sweep.json in the same shape as
# results/classifier_baselines.json (TF-IDF + Claude Haiku).
!python -m mostargate.classifier.sweep

## 7. Bottleneck check - sweep with `email_send_external` handled by an oracle

Per `docs/phase_c_classifier_findings.md` §7.8, `email_send_external` is the classifier-hardest permission *and* the trifecta-completing one. The CLI's `--exclude` flag models the hybrid architecture: assume an external deterministic rule (recipient-domain check) decides this permission perfectly, replacing the classifier's prediction on it with ground truth. The 14 other permissions are still classifier-handled.

Mathematically: the excluded permission contributes zero overshoot/undershoot, and the metrics stay in the 15-permission space so they compare directly with the TF-IDF and Claude Haiku baselines. The run is merged into the same JSON file under a different condition key (`finetuned_<model>_oracle_email_send_external`).

In [ ]:
!python -m mostargate.classifier.sweep --exclude email_send_external

## 8. Zip the model + cached probs for download

The zip contains only the **gitignored** artifacts — the trained model directory and the cached prediction matrix. The metrics JSON (written by cells 6 and 7) lives at `results/<short_model>_sweep.json`, tracked in git, downloaded separately.

You'll download **two things** from Colab:

1. `classifier_model_<short_model>.zip` (~325 MB for base, ~1.4 GB for large) — unzip locally so files land under `dataset/classifier_artifacts/`. Don't commit.
2. `results/<short_model>_sweep.json` (~150 KB) — drop in your local `results/` directory. Commit alongside `classifier_baselines.json` so the run is reproducible from git alone.

In [ ]:
# Zip only the model directory and the cached prediction matrix.
# The sweep_results.json is written to results/ (a tracked-in-repo dir),
# NOT into this zip — download the JSON separately so it can be committed
# to the repo alongside the rest of the metrics archives.
#
# Output filename includes the base model name so base/large runs don't
# clobber each other on download. We read the name from model_card.md
# rather than config.json: HF transformers doesn't always include
# `_name_or_path` in config.json, but train.py reliably writes the
# canonical name to `Base model: \`<model_name>\`` in the card.
import re
from pathlib import Path

card_text = Path("dataset/classifier_artifacts/model/model_card.md").read_text()
m_name = re.search(r"Base model:\s*`([^`]+)`", card_text)
model_name = m_name.group(1) if m_name else "unknown"
short_model = model_name.split("/")[-1].replace("-", "_")
zip_name = f"classifier_model_{short_model}.zip"

!zip -qr {zip_name} \
    dataset/classifier_artifacts/model \
    dataset/classifier_artifacts/roberta_test_probs.npy
!ls -lh {zip_name}

print(f"\nDownload TWO files from the Files panel on the left:")
print(f"  1. {zip_name}  (~325–1400 MB depending on base vs large)")
print(f"  2. results/{short_model}_sweep.json  (~150 KB, commit this to the repo)")